In [ ]:
# --- SETUP ---
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time, copy
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)

Running on: cpu


In [ ]:
#DATASET LOADING
selected_n_value = 16

X = np.load('datasets/kryptonite-%s-X.npy'%(selected_n_value))
len(X)
y = np.load('datasets/kryptonite-%s-y.npy'%(selected_n_value))

X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.long)

# Train/val/test split
n = len(X)
train_size = int(0.7*n)
val_size = int(0.15*n)
test_size = n - train_size - val_size
train_ds, val_ds, test_ds = random_split(TensorDataset(X,y), [train_size, val_size, test_size])

train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256)
test_dl = DataLoader(test_ds, batch_size=256)


In [115]:
class MLP(nn.Module):
    # randomly drops 30% of neurons during training to keep the model from overfitting
    def __init__(self, in_dim, hidden=[512,256,128], dropout=0.3):
        super().__init__()
        layers = []
        last = in_dim
        for h in hidden:
            layers += [nn.Linear(last, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            last = h
        layers.append(nn.Linear(last, 2))  # Binary classification
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [116]:
def train_one_epoch(model, opt, loader):
    model.train()
    total_loss, grad_norms = 0.0, []
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = model(xb)
        loss = F.cross_entropy(out, yb)
        loss.backward()
        total_g = torch.sqrt(sum((p.grad**2).sum() for p in model.parameters())).item()
        grad_norms.append(total_g)
        opt.step()
        total_loss += loss.item() * xb.size(0)
    return total_loss / len(loader.dataset), np.mean(grad_norms)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, correct = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = F.cross_entropy(out, yb)
        total_loss += loss.item() * xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)


In [117]:
def run_experiment(config):
    torch.manual_seed(config['seed'])
    model = MLP(in_dim=config['in_dim'], hidden=config['hidden']).to(device)
    if config['init'] == 'he':
        for m in model.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
    opt = torch.optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['wd'])

    
    history = {'train_loss':[], 'val_loss':[], 'val_acc':[], 'grad_norm':[]}
    best = {'val_acc': -1.0, 'state_dict': None, 'epoch': -1}

    early_stopping_tolerance = 10
    no_loss_update = 0

    for epoch in range(config['epochs']):
        tr_loss, gnorm = train_one_epoch(model, opt, config['train_dl'])
        val_loss, val_acc = evaluate(model, config['val_dl'])
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['grad_norm'].append(gnorm)

        if val_acc > best['val_acc']:
            best['val_acc'] = val_acc
            best['epoch'] = epoch
            best['state_dict'] = {k: v.cpu() for k, v in model.state_dict().items()}
            no_loss_update = 0
        else:
            if config['early_stopping_cond']:
                no_loss_update += 1
                if no_loss_update >= early_stopping_tolerance:
                    break
         
    # load best for test
    model.load_state_dict(best['state_dict'])
    model.to(device)
    test_loss, test_acc = evaluate(model, config['test_dl'])


    return  {
        'config': {k: (v if k not in ('train_dl','val_dl','test_dl') else f'<{k}>') for k,v in config.items()},
        'history': history,
        'best_epoch': best['epoch'],
        'test_loss': test_loss,
        'test_acc': test_acc,
        'state_dict': best['state_dict'],
    }
    


In [118]:
# Grid of LRs and batch sizes for convergence analysis -- CHOOSE THE LEARNING RATE (lr) AND BATCH SIZE (bs)
configs = []
seeds = [1, 2, 3]

if n == 10 or n == 12:
    print("Early Stopping")
    early_stopping_bool = True
else:
    early_stopping_bool = False

for lr in [1e-3, 3e-3, 1e-2]:
    for bs in [64, 256]:
        for seed in [1, 2, 3]:
            configs.append({
                'in_dim': X.shape[1],
                'hidden': [512,256,128],
                'dropout': 0.3,
                'lr': lr,
                'wd': 1e-4,
                'init': 'he',
                'epochs': 80,
                'train_bs': bs,
                'train_dl': DataLoader(train_ds, batch_size=bs, shuffle=True),
                'val_dl': val_dl,
                'test_dl': test_dl,
                'seed': seed,
                'early_stopping_cond': early_stopping_bool
            })

results = []

for cfg in configs:
    start = time.time()
    experiment_results = run_experiment(cfg)
    experiment_results['time'] = time.time()-start
    results.append(experiment_results)
    print(f"LR={cfg['lr']}, BS={cfg['train_dl'].batch_size}, Seed={cfg['seed'], }Test acc={experiment_results['test_acc']:.3f}")


# Get the index of the best model ie the one with highest validation accuracy
best_idx = int(np.argmax([r['history']['val_acc'][int(r['best_epoch'])] for r in results]))
best_run = results[best_idx]


results_to_save = {
    'n_features': selected_n_value,
    'results': [
        {
            'config': r['config'],
            'history': r['history'],
            'test_loss': r['test_loss'],
            'test_acc': r['test_acc'],
        } for r in results
    ],
    'best_idx': best_idx, 
    'best_state_dict': best_run['state_dict'],
}

torch.save(results_to_save, f'models/results_{selected_n_value}')


LR=0.001, BS=64, Seed=(1,)Test acc=0.929
LR=0.001, BS=64, Seed=(2,)Test acc=0.920
LR=0.001, BS=64, Seed=(3,)Test acc=0.945
LR=0.001, BS=256, Seed=(1,)Test acc=0.914
LR=0.001, BS=256, Seed=(2,)Test acc=0.916
LR=0.001, BS=256, Seed=(3,)Test acc=0.914
LR=0.003, BS=64, Seed=(1,)Test acc=0.711
LR=0.003, BS=64, Seed=(2,)Test acc=0.496
LR=0.003, BS=64, Seed=(3,)Test acc=0.496
LR=0.003, BS=256, Seed=(1,)Test acc=0.829
LR=0.003, BS=256, Seed=(2,)Test acc=0.845
LR=0.003, BS=256, Seed=(3,)Test acc=0.900
LR=0.01, BS=64, Seed=(1,)Test acc=0.504
LR=0.01, BS=64, Seed=(2,)Test acc=0.496
LR=0.01, BS=64, Seed=(3,)Test acc=0.496
LR=0.01, BS=256, Seed=(1,)Test acc=0.504
LR=0.01, BS=256, Seed=(2,)Test acc=0.496
LR=0.01, BS=256, Seed=(3,)Test acc=0.504
